# Simulating Robot Systems with MuJoCo

This notebook introduces the MuJoCo-based simulation workflow in RobotBlockSet. Its purpose is to show how to simulate robot systems with the MuJoCo backend, including robot creation, scene setup, and execution of motion-related examples.


## What this notebook covers

The examples below demonstrate how RobotBlockSet connects robot models and scenes to the MuJoCo simulator, how robot objects are created for simulation, and how the resulting setup can be used for motion execution, visualization, sensing, and controller-related experiments.

Use this notebook as a practical starting point when you want to run robot-system simulations with MuJoCo or adapt the same workflow to your own MJCF models and simulated environments.


# Imports


In [1]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from datetime import datetime

from robotblockset.tools import get_rbs_path

np.set_printoptions(formatter={"float": "{: 0.4f}".format})

# Robot classes for MuJoCo simulator

MuJoCo (Multi-Joint dynamics with Contact) is a high-performance physics engine widely used for robotics, biomechanics, and reinforcement learning. It provides accurate and efficient simulation of articulated mechanisms, contacts, and sensors, making it an excellent backend for robotic experiments and algorithm development. Robot models in MuJoCo are described in XML using the MJCF modeling language, which defines kinematics, dynamics, actuators, and sensors.

RBS integrates MuJoCo in three different ways, giving users flexibility in how simulations are executed and controlled:

1. **mujoco** - *Standalone MuJoCo application* – RBS communicates with `simmujoco` application, which is an extended version of the original `simulate` program, augmented with a TCP server. This allows RBS to communicate with the simulator: read robot states and sensor data, access object positions, control actuators, and reposition models in the scene. To build the `simmujoco` application see [README.md](/d:/Leon/Python/RBS/robotblockset/mujoco/simmujoco/README.md).
2. **pymujoco** - *Mujoco with Native Python bindings* – RBS interacts directly with the MuJoCo engine and Viewer through Python bindings, where the simulator runs independently in a separate thread.
3. **pymujoco_sim** - *RBS-controlled integration Mujoco with Native Python bindings* – similar to the Python binding approach, but with simulation stepping fully managed by RBS, enabling precise synchronization and also supporting headless (non-visual) simulation.

These options allow RBS to adapt MuJoCo to a wide range of use cases, from interactive visual debugging to fully automated, large-scale simulation workflows.

The robot models for MuJoCo simulator are defined in a XML using  MJCF modelling language. For details see https://mujoco.readthedocs.io/en/stable/modeling.html. Additinally, the tutorial `tutorial_generate_MJCF_scene` describes how to programatically generate complex MJCF models

All common specific parameters and methods for MuJoCo target are defined for MuJoCo in subclasses: 
- `robot_mujoco.py` for target **mujoco**
- `robot_pymujoco.py` for target **pymujoco**
- `robot_pymujoco_sim.py` for target **pymujoco_sim**

To control a robot model in MuJoCo using RBS Toolbox we have to send to the MuJoCo simulator commands to actuators and get the information about the robot reading the robot states and sensors. Currently, all robot joints in MuJoCo simulator have to be connected to actuators, using internal position control. Therefore, the MuJoCo targets support only *joint position control strategy*

> **Note:** It is possible to define cunstom control algorithms using targets `robot_pymujoco.py` and `robot_pymujoco_sim.py`. In this case the sensors and actuators should be properly defined in MJCF mode. allow also to define custom controllers, but in this case

# Creation of a robot object

The creation of a robot object depends on the target robot platform. 

## Target **mujoco**

For target **mujoco** we have to use `simmujoco`, which is an extended version of the MuJoCo application `simulate`. 

First we have to load the corresponding MJCF model into `simmujoco`. Then we define the robot object.

### Manual opening of scene

If we open `simmujoco` application directly, then we have to load the MJCF model by dropping the corresponding model file (`<model>.xml`) in `simmujoco`.


In [14]:
from robotblockset.mujoco.robots_mujoco import panda

r = panda(robot_name="panda")

[RBS_INFO] [1773401731.547704935] [panda_MuJoCo]: Robot connected to MuJoCo


> ⚠️ **Warning:** The `robot_name` must equal the name of the first body in the robot body tree.

### Programatically loading the scene

The other possibility is to start the applicaytion from Python. For example, start MuJoCo simulator on **localhost** and non-defualt **port** (to prevent connection to a possible opened simulator).

> **Note:** The path to `simmujoco` in the `subprocess.Popen(...)` call must match your local installation. If `simmujoco` was built or installed in a different location, update the path accordingly. The example below uses the executable from the local RBS folder returned by `get_rbs_path()`.

In [3]:
import subprocess

p = subprocess.Popen([get_rbs_path() + "/mujoco/bin/win64/bin/simmujoco.exe", "localhost", "50001"])

Manually open connection to simulator by defining the scene at the selected port

In [4]:
from robotblockset.mujoco.mujoco_api import mjInterface as mujoco_scene

scene = mujoco_scene(port=50001)
scene.mj_connect()

0

Load scene in MuJoCo simulator

In [ ]:
from time import sleep

scene.mj_load(get_rbs_path() + "/mujoco/mjcf_models/panda_scene.xml")
sleep(2)

Finally, we create the robot object

In [7]:
from robotblockset.mujoco.robots_mujoco import panda

r = panda(robot_name="panda", scene=scene)

[RBS_INFO] [1773401705.203033686] [panda_MuJoCo]: Robot connected to MuJoCo


## Target **pymujoco** or **pymujoco_sim** 

For the targets **pymujoco** or **pymujoco_sim** we have to define first the scene

In [8]:
from robotblockset.mujoco.robots_pymujoco import mujoco_scene, panda

MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/panda_scene.xml"
scene = mujoco_scene(MODEL_PATH, show_camera=None, verbose=1)

Arguments for `mujoco_scene`:

| Argument | Default | Description |
| --- | --- | --- |
| `model_xml_file` | `None` | Path to the MuJoCo model XML file to load. |
| `model` | `None` | Existing `mujoco.MjModel` instance; used instead of loading from XML. |
| `show_viewer` | `True` | It `True`, viewer is opened (only for **pymujoco_sim**). |
| `show_camera` | `None` | List of camera IDs or names to display in separate windows; `None` shows no extra camera windows. |
| `synchronized` | `True` | If `True`, simulation steps are synchronized to a fixed timestep (only for **pymujoco**). |
| `verbose` | `0` | Verbosity level for debug messages. |


Then we can define a robot object

In [9]:
r = panda(scene=scene)

[RBS_INFO] [1773401706.198982954] [panda_PyMuJoCo]: Robot connected to MuJoCo


Arguments for `robot_mujoco` / `robot_pymujoco` (union across `robots_mujoco.py`, `robots_pymujoco.py`, `robots_pymujoco_sim.py`):

| Argument | Default | Description |
| --- | --- | --- |
| `robot_name` | required | Robot name used for naming and model lookups. |
| `scene` | `None` | MuJoCo scene/connection object; required for `robot_pymujoco` and optional for `robot_mujoco` (creates its own interface if `None`). |
| `host` | `"localhost"` | MuJoCo server host (only in `robot_mujoco`). |
| `port` | `50000` | MuJoCo server port (only in `robot_mujoco`). |
| `**kwargs` | ? | Additional keyword args (see below). |



## Definition of model handels

To be able to read the states and sensors, and to send control signals, it is necessary to know the indices of MuJoCo internal variables, or handles of MuJoCo object. 

The simplest way is to define in MJCF model names robot objects according to the name scheme given in the table, where object name is a combination of the robot name `<robot>` defined during the robot object creation, followed by the separator `_` and the descriptive name of the object, and in some cases with a numer as postfix.

| Property             | Defualt MuJoCo name        | Description                                                  |
| -------------------- | -------------------------- | :----------------------------------------------------------- |
| `robot_name`         | `<robot>`                  | Base link name, i.e., the first body in MuJoCo robot body tree (defines robot base frame)    |
| `JointNames`         | `<robot>_joint<i>`     | Joint names in MuJoCo robot model                                  |
| `ActuatorNames`      | `<robot>_pos_joint<i>` | Actuator names for joint position control in MuJoCo robot model    |
| `FlangeName`             | `<robot>_flange`           | Site name attached to the last link representing the end-effector (used to calculate TCP transformation) |
| `TCPName`            | `<robot>_TCP`         | Site name at the TCP location (used to calculate TCP transformation) |
| `SensorJointPosName` | `<robot>_pos_joint<i>` | Joint position sensor names in MuJoCo model                  |
| `SensorJointVelName` | `<robot>_vel_joint<i>` | Joint velocity sensor names in MuJoCo model                  |
| `SensorPosName`      | `<robot>_pos`          | EE position sensor name in MuJoCo model                      |
| `SensorOriName`      | `<robot>_ori`          | EE orientation sensor name in MuJoCo model                   |
| `SensorLinVelName`   | `<robot>_v`            | EE linear velocity sensor name in MuJoCo model               |
| `SensorRotVelName`   | `<robot>_w`            | EE rotational velocity sensor name in MuJoCo model           |
| `SensorForceName`    | `<robot>_force`        | EE force sensor name in MuJoCo model                         |
| `SensorTorqueName`   | `<robot>_torque`       | EE torque sensor name in MuJoCo model                        |

> **Note:** Joint positions and velocities are updated by default using joint sensors defined by `SensorJointPosName` and `SensorJointVelName`. If these sensors are not defined in the model, then joint positions and velocities are read form the model state using indices obtaind by using `JointNames`.

A typical definition of a robot in MuJoCo model definition file looks like this
```xml
<body name="panda" childclass="panda_robot">
  <body name="panda_link0">
    <inertial pos="-0.041018 -0.00014 0.049974" quat="0.00630474 0.751245 0.00741774 0.659952" mass="0.629769" diaginertia="0.00430465 0.00387984 0.00313051"/>
    <geom  ... />
    <body name="panda_link1" pos="0 0 0.333">
      <inertial pos="0.003875 0.002081 -0.04762" quat="0.711549 0.00634377 -0.0131124 0.702485" mass="4.97068" diaginertia="0.707137 0.703435 0.00852456"/>
      <joint name="panda_joint1" pos="0 0 0" axis="0 0 1"/>
      <geom ... />
      <body name="panda_link2" quat="0.707107 -0.707107 0 0">
      ...
      </body>
    </body>
  </body>
</body>
```

A typical definition of actuators in MuJoCo model definition file looks like this

```xml
<actuator>
  <general name="panda_pos_joint1" class="panda_robot" joint="panda_joint1" gainprm="4500" biasprm="0 -4500 -450"/>
  <general name="panda_pos_joint2" class="panda_robot" joint="panda_joint2" ctrlrange="-1.7628 1.7628" gainprm="4500" biasprm="0 -4500 -450"/>
  <general name="panda_pos_joint3" class="panda_robot" joint="panda_joint3" gainprm="3500" biasprm="0 -3500 -350"/>
  <general name="panda_pos_joint4" class="panda_robot" joint="panda_joint4" ctrlrange="-3.0718 -0.0698" gainprm="3500" biasprm="0 -3500 -350"/>
  <general name="panda_pos_joint5" class="panda_robot" joint="panda_joint5" forcerange="-12 12" gainprm="2000" biasprm="0 -2000 -200"/>
  <general name="panda_pos_joint6" class="panda_robot" joint="panda_joint6" ctrlrange="-0.0175 3.7525" forcerange="-12 12" gainprm="2000" biasprm="0 -2000 -200"/>
  <general name="panda_pos_joint7" class="panda_robot" joint="panda_joint7" forcerange="-12 12" gainprm="2000" biasprm="0 -2000 -200"/>
  <general name="panda_tool_panda_hand" class="panda_tool_panda_hand" tendon="panda_tool_split" ctrlrange="0 255" forcerange="-100 100" gainprm="0.0156863" biasprm="0 -100 -10"/>
</actuator>
```

A typical definition of sensors in MuJoCo model definition file looks like this

```xml
<sensor>
	<framepos name="panda_pos" objtype="site" objname="panda_hand" />
	<framequat name="panda_ori" objtype="site" objname="panda_hand" />
	<framelinvel name="panda_v" objtype="site" objname="panda_hand" />
	<frameangvel name="panda_w" objtype="site" objname="panda_hand" />
	<force name="panda_force" site="panda_hand" noise="0.5" />
	<torque name="panda_torque" site="panda_hand" noise="0.5" />
</sensor>
```

```python
r.JMove(q, t, traj="trap")
r.JLine(q, t)
```

If the MuJoCo robot model does not use the default object names, then there are other posibilities:
- the object names are defined when creating a robot object by using the corresponding keyword arguments. 

```python
    r = panda(robot_name="panda", TCPName="panda_gripper")
```

- the object names are defined in specification class of the robot

```python
    class ur10_spec(robot):
        def __init__(self) -> None:
            self.Name = "ur10"
            self.nj = 6
            ...
            self.joint_names = [
                "shoulder_pan_joint",
                "shoulder_lift_joint",
                "elbow_joint",
                "wrist_1_joint",
                "wrist_2_joint",
                "wrist_3_joint",
            ]
```

- when using **pymujoco** or **pymujoco_sim**, the constructor arguments `JointNames` and `ActuatorNames` can also be used to discover or generate the required names directly from the MJCF model.

Meaning of `JointNames`:

- `JointNames="auto"` means that the backend searches the MJCF model under the robot base body and automatically collects the first `nj` robot joints from the model tree. This is useful when the model uses custom joint names and you want to read them directly from the loaded MJCF model.
- `JointNames="gen"` means that the backend does not inspect the MJCF model for joint names, but generates them using the default naming rule `<robot>_joint1`, `<robot>_joint2`, ..., `<robot>_joint<nj>`.
- `JointNames=None` means that the backend first tries to use `self.joint_names` from the robot specification class, and if these names are not available it falls back to the generated default names `<robot>_joint<i>`.

Meaning of `ActuatorNames`:

- `ActuatorNames="auto"` means that the backend finds actuators directly from the MJCF model for the already selected robot joints.
- `ActuatorNames=None` means that actuator names are generated from `JointNames` by using the default position-actuator naming rule `<robot>_pos_joint<i>`.
- In these backends there is no `ActuatorNames="gen"` keyword. Automatic discovery is available through `"auto"`, otherwise the names are either generated from `JointNames` or given explicitly as a list.

Typical examples:

```python
    r = panda(scene=scene, JointNames="auto", ActuatorNames="auto")
```

This is the most robust option when the MJCF model already contains the correct joint and actuator definitions and you want the backend to read them directly from the model.

```python
    r = panda(scene=scene, JointNames="gen")
```

This is useful when the generated scene follows the standard naming convention `<robot>_joint<i>`. In this case, actuator names are by default generated as `<robot>_pos_joint<i>`.

```python
    r = ur10e(scene=scene)
```

In this case `JointNames` is not given explicitly, so the backend uses the names defined in the robot specification class if available.

In short, for **pymujoco** and **pymujoco_sim**:
- use `"auto"` when names should be read from the MJCF model,
- use `"gen"` only for `JointNames` when the standard generated joint naming rule is used,
- use explicit lists or robot specifications when the model uses a custom naming scheme.


A typical definition of sensors in MuJoCo model definition file looks like this

```xml
<sensor>
  <framepos name="panda_pos" objtype="site" objname="panda_hand" />
  <framequat name="panda_ori" objtype="site" objname="panda_hand" />
  <framelinvel name="panda_v" objtype="site" objname="panda_hand" />
  <frameangvel name="panda_w" objtype="site" objname="panda_hand" />
  <force name="panda_force" site="panda_hand" noise="0.5" />
  <torque name="panda_torque" site="panda_hand" noise="0.5" />
</sensor>
```

# MuJoCo specific robot methods

The `robot_mujoco.py`, `robot_pymujoco.py` and `robot_pymujoco_sim.py` classes define some for MuJoCo target specific methods


| Method | Description |
| ------ | ----------- |
| `SendRobot_u(u)` | Send joint commands to robot actuators. |
| `SendCtrl(u)` | Send a control vector to all actuators. |
| `SendAuxCtrl(idx, val)` | Update selected actuator controls by index. |
| `GetAuxJointPos(idx)` | Return joint positions for auxiliary joints by index. |
| `GetSensor(name_or_id=None)` | Read sensor data by name/id or return the full sensor array. |
| `GetContacts(name_or_id=None)` | Return contact forces for a geom or for all contacts. |
| `GetMocapPose(name_or_id, out="x")` | Return mocap body pose in the requested output format. |
| `SetRobotPose(x)` | Set pose of a robot. |
| `SetMocapPose(name_or_id, x)` | Set pose of a mocap body. |
| `GetObjectData(name_or_id)` | Return raw MuJoCo body data for a body name/id. |
| `GetObjectPose(typ, name_or_id, out="x")` | Return the pose of a body/site/geom in the requested format. |
| `SetObjectPose(name_or_id, x)` | Set a body pose. |
| `SetEquality(name_or_id, active)` | Set an equality constraint activation flag. |
| `UpdateRobotBaseFromModel()` | Update the cached robot base pose from the MuJoCo model. |
| `SimulatorMessage(msg)` | Send a message to the simulator UI. |
| `sim(dt)` | Advance the simulator for a fixed duration. |

Classes `robot_pymujoco.py` and `robot_pymujoco_sim.py` define additionally some properties

| Property | Description |
| --- | --- |
| `J` | Jacobian matrix J(q). |
| `H` | Inertia matrix H(q) in joint space. |
| `M` | Inertia matrix M(q) in task space. |
| `c` | Coriolis, centrifugal, and gravitational joint torques. |

For example to get the Jacobian matrix in some configuration, you can use

In [10]:
print("Jacobian matrix using RBS model:\n", r.Jacobi(r.q_home))

Jacobian matrix using RBS model:
 [[ 0.0000  0.1539  0.0000  0.1279  0.0000  0.2104  0.0000]
 [ 0.3069  0.0000  0.3258  0.0000  0.2104  0.0000 -0.0000]
 [ 0.0000 -0.3069  0.0000  0.4720  0.0000  0.0880  0.0000]
 [ 0.0000 -0.0000 -0.7071  0.0000  1.0000  0.0000  0.0000]
 [ 0.0000  1.0000 -0.0000 -1.0000  0.0000 -1.0000  0.0000]
 [ 1.0000  0.0000  0.7071  0.0000 -0.0000  0.0000 -1.0000]]


or to get it from MuJoCo use

In [11]:
r.JMove(r.q_home)
print("Jacobian matrix from MuJoCo model:\n", r.J)

Jacobian matrix from MuJoCo model:
 [[ 0.0000  0.1560  0.0000  0.1270 -0.0000  0.2112  0.0000]
 [ 0.3066  0.0000  0.3279  0.0001  0.2116  0.0001 -0.0000]
 [ 0.0000 -0.3066  0.0000  0.4697 -0.0001  0.0861 -0.0000]
 [ 0.0000 -0.0000 -0.7019  0.0001  1.0000  0.0001 -0.0091]
 [ 0.0000  1.0000 -0.0000 -1.0000  0.0001 -1.0000 -0.0005]
 [ 1.0000  0.0000  0.7122  0.0001  0.0045  0.0005 -1.0000]]


> **Note:** To get properties from MuJoCo model, the robot has to be moved to desired configuration.

For example, to get robot inertia matrix use

In [12]:
print("Inertia matrix intask space:\n", r.M)

Inertia matrix intask space:
 [[ 12.4345 -0.1935  1.5874  0.0425  2.4535  0.0002]
 [-0.1935  5.1388 -0.0350 -0.8099 -0.0440 -0.1954]
 [ 1.5874 -0.0350  6.5677  0.0672  0.9012 -0.0015]
 [ 0.0425 -0.8099  0.0672  0.2528  0.0139  0.0354]
 [ 2.4535 -0.0440  0.9012  0.0139  0.6847 -0.0017]
 [ 0.0002 -0.1954 -0.0015  0.0354 -0.0017  0.1001]]


We can position a mocap object in the model to a specific pose. For example, we can position a dummy body **Target** (defined in many RBS models) to a specific location by using command:

In [13]:
r.SetMocapPose("Target", [0.5, 0.2, 0.6])

> ℹ️ **Note:** Mujoco object can be  grouped in different groups from 0 to 5. If you do not see the object, check the group visibility in simulator.

> ℹ️ **Note:** Using classes `robot_pymujoco.py` and `robot_pymujoco_sim.py` a lot of model parameters and object parameters can be get or set. For mor details see [MuJoCo documentation](https://mujoco.readthedocs.io/en/stable/python.html)
